In [4]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
import joblib
import os

# 1. Create the directory for saving trained models
os.makedirs('../src/models', exist_ok=True)

# 2. Load and prepare the dataset
df = pd.read_csv('../data/processed/processed_data.csv').astype(float)
X = df.drop('is_long_stay', axis=1)
y = df['is_long_stay']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline model: predicts the class distribution observed in the training set
model_baseline = DummyClassifier(strategy='prior')
model_baseline.fit(X_train, y_train)

# Advanced model: XGBoost classifier for the final comparison
model_advanced = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)
model_advanced.fit(X_train, y_train)

# 3. Compare model performance for reporting
base_auc = roc_auc_score(y_test, model_baseline.predict_proba(X_test)[:, 1])
adv_auc = roc_auc_score(y_test, model_advanced.predict_proba(X_test)[:, 1])

print(f"Baseline model AUC: {base_auc:.4f}")
print(f"Advanced model AUC: {adv_auc:.4f}")

# 4. Save models and feature names for later use in the microservice
joblib.dump(model_baseline, '../src/models/model_baseline.joblib')
joblib.dump(model_advanced, '../src/models/model_advanced.joblib')
joblib.dump(X.columns.tolist(), '../src/models/model_features.joblib')

print("Models have been saved to ../src/models/")


Baseline model AUC: 0.5000
Advanced model AUC: 0.9635
Models have been saved to ../src/models/
